# Setting

In [3]:
!pip install ginkgo_ai_client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 32.3 MB/s eta 0:00:00


In [4]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ginkgo_ai_client import GinkgoAIClient, DiffusionMaskedQuery

from Bio import AlignIO
from Bio.Seq import Seq
from Bio.Align import MultipleSeqAlignment
from Bio.SeqRecord import SeqRecord
from google.colab import userdata

In [5]:
# set environment variable GINKGOAI_API_KEY
GINKGO_API_KEY = "df3b0629-c981-4fe3-8acb-4968ed50716e"  # from ginkgo's colab example
client = GinkgoAIClient(api_key = GINKGO_API_KEY)

# Define Ab

In [6]:
# Define heavy and light sequence to evaluate
ab_heavy_seq = "EVQLLESGGDLVQPGGSLKLSCAASGFTFSNYAINWVRQAPGKGLEWVSTISNSGDSTVHADSVRGRFTISRDNSKSTLYLQMNSLRVEDTAVYFCAKGQLPYCTNWRCYAFDYWGQGTLVTVSS"
ab_light_seq = "EIVMTQSPATLSVSPGERATLSCRASQSVNSNLAWYQQKPGQALRLLIYGASTRATGIPARFSGSGSGTEFTLTISSLQSEDFAVYYCQQYNHWGTFGQGTKVEIK"

# Define CDRs for evaluation of masking
ab_heavy_cdr3 = "CAKGQLPYCTNWRCYAFDYW"
ab_light_cdr3 = "CQQYNHWGTF"

In [7]:
# Replace antibody chain CDRs with masking
ab_heavy_masked_seq = ab_heavy_seq.replace(ab_heavy_cdr3, "".join(["<mask>"]*len(ab_heavy_cdr3)))
ab_light_masked_seq = ab_light_seq.replace(ab_light_cdr3, "".join(["<mask>"]*len(ab_light_cdr3)))
print(f"Masked positions sequence:\nHeavy - {ab_heavy_masked_seq}\nLight - {ab_light_masked_seq}")

Masked positions sequence:
Heavy - EVQLLESGGDLVQPGGSLKLSCAASGFTFSNYAINWVRQAPGKGLEWVSTISNSGDSTVHADSVRGRFTISRDNSKSTLYLQMNSLRVEDTAVYF<mask><mask><mask><mask><mask><mask><mask><mask><mask><mask><mask><mask><mask><mask><mask><mask><mask><mask><mask><mask>GQGTLVTVSS
Light - EIVMTQSPATLSVSPGERATLSCRASQSVNSNLAWYQQKPGQALRLLIYGASTRATGIPARFSGSGSGTEFTLTISSLQSEDFAVYY<mask><mask><mask><mask><mask><mask><mask><mask><mask><mask>GQGTKVEIK


In [8]:
ab_heavy_masked_seq

'EVQLLESGGDLVQPGGSLKLSCAASGFTFSNYAINWVRQAPGKGLEWVSTISNSGDSTVHADSVRGRFTISRDNSKSTLYLQMNSLRVEDTAVYF<mask><mask><mask><mask><mask><mask><mask><mask><mask><mask><mask><mask><mask><mask><mask><mask><mask><mask><mask><mask>GQGTLVTVSS'

Temperature can be betweeen 0.0 and 1.0 and controls the variability of the mask infilling. Higher values result in more variable and, potentially, natural sampling. Decoding strategy can either be `"max_prob"` or `"entropy"` and these tune the strategy to decide which positions to decode. Lastly, unmaskings_per_step controls how many positions ot unmask each pass through the model. This can speed up generation while potentially lowering the sample quality. We recommend a value of `1.0` for temperature, `"entropy"` for decoding_order_strategy, and `4` for unaskings_per_step as defaults. Feel free to play around and see how the results change! For more info refer the reader to the provided write-up: HERE.

In [ ]:
# Fill in positions with diffusion generation uisng the API
model = "abdiffusion"
temperature = 1.0
decoding_order_strategy = "entropy"
unmaskings_per_step = 4
query = DiffusionMaskedQuery(
    sequence=ab_heavy_masked_seq,
    unmaskings_per_step=unmaskings_per_step,
    model=model,
    temperature=temperature,
    decoding_order_strategy=decoding_order_strategy,
    query_name="Unconditional diffusion generation of CDR3 of heavy antibody string")
response = client.send_request(query, timeout=1200)

In [ ]:
response

In [ ]:
# Light chain unmasking
query = DiffusionMaskedQuery(
    sequence=ab_light_masked_seq,
    unmaskings_per_step=unmaskings_per_step,
    model="abdiffusion",
    temperature=temperature,
    decoding_order_strategy=decoding_order_strategy,
    query_name="Unconditional diffusion generation of CDR3 of light antibody string")
response = client.send_request(query, timeout=1200)

In [ ]:
response